# Ejercicio: estimación aproximada de edad con cámara

Este notebook captura una imagen autorizada por el usuario y estima un rango de edad aparente a partir del rostro usando DeepFace.

> La estimación puede tener errores y sesgos. No es una edad oficial, no identifica a la persona y no debe utilizarse para decisiones laborales, legales, médicas, de acceso o vigilancia. Use únicamente imágenes con consentimiento.

## 1. Instalar DeepFace

Se instala DeepFace `0.0.100` y se importa el módulo `DeepFace`. La primera ejecución puede tardar porque Colab descarga dependencias y el modelo de edad.

**Resultado esperado:** deben aparecer las versiones instaladas sin errores.

In [ ]:
!pip -q install deepface==0.0.100

import deepface
from deepface import DeepFace
import base64
import cv2
import matplotlib.pyplot as plt
from google.colab import output
from IPython.display import Javascript, display

print('DeepFace:', deepface.__version__)
print('OpenCV:', cv2.__version__)

## 2. Activar cámara y capturar una imagen

Colab funciona en un servidor remoto, por lo que la cámara se abre mediante JavaScript en el navegador. Al ejecutar la celda, acepte el permiso de cámara. Se espera 2.5 segundos para que la imagen se estabilice, captura un cuadro y detiene la cámara.

**Recomendaciones:** rostro de frente, buena iluminación, sin lentes oscuros ni objetos que cubran la cara. La imagen se guarda solo temporalmente en `/content` durante la sesión.

In [ ]:
display(Javascript(r'''
async function captureAgeFrame() {
  const video = document.createElement('video');
  video.width = 640; video.height = 480; video.autoplay = true;
  video.style.border = '3px solid #1f77b4';
  document.body.appendChild(video);
  const stream = await navigator.mediaDevices.getUserMedia({video: true, audio: false});
  video.srcObject = stream;
  await new Promise(resolve => setTimeout(resolve, 2500));
  const canvas = document.createElement('canvas');
  canvas.width = video.videoWidth || 640; canvas.height = video.videoHeight || 480;
  canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);
  stream.getTracks().forEach(track => track.stop());
  video.remove();
  return canvas.toDataURL('image/jpeg', 0.85);
}
'''))

image_data = output.eval_js('captureAgeFrame()')
image_bytes = base64.b64decode(image_data.split(',')[1])
image_path = '/content/captura_edad.jpg'
with open(image_path, 'wb') as image_file:
    image_file.write(image_bytes)
print('Captura lista:', image_path)

## 3. Estimar edad aparente

`DeepFace.analyze` recibe la imagen y ejecuta la acción `age`. `enforce_detection=True` detiene el análisis si no encuentra un rostro confiable, lo cual evita presentar una estimación basada en una imagen incorrecta.

El campo `age` es una estimación numérica del modelo; debe tratarse como aproximada. Para una interpretación más prudente, se muestra también un intervalo de ±5 años.

In [ ]:
try:
    analysis = DeepFace.analyze(
        img_path=image_path,
        actions=['age'],
        enforce_detection=True,
        detector_backend='opencv',
        silent=True
    )
except Exception as error:
    raise RuntimeError('No se detectó un rostro claro. Mejore la iluminación y repita la captura.') from error

if isinstance(analysis, list):
    analysis = analysis[0]

estimated_age = float(analysis['age'])
lower_age = max(0, round(estimated_age - 5))
upper_age = round(estimated_age + 5)
image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.axis('off')
plt.title(f'Edad aparente estimada: {estimated_age:.0f} años')
plt.show()

print(f'Edad aparente estimada: {estimated_age:.1f} años')
print(f'Rango orientativo para el ejercicio: {lower_age}–{upper_age} años')
print('La estimación no sustituye un documento oficial ni una evaluación profesional.')

## 4. Interpretar resultados y repetir

Si la salida indica 30 años, no significa que la edad real sea exactamente 30. Significa que el modelo encontró patrones visuales compatibles con esa edad aparente. El rango de ±5 años es una forma didáctica de comunicar incertidumbre; no es un intervalo estadístico calibrado.

Para repetir el ejercicio, ejecute de nuevo la celda de cámara y después la celda de análisis. Compare cómo cambian los resultados con iluminación, distancia, expresión facial y ángulo.

### Preguntas

1. ¿Cuánto cambia la estimación entre tres capturas?
2. ¿Qué efecto tienen lentes, barba, maquillaje o una imagen de perfil?
3. ¿Por qué la edad aparente no debe usarse como edad legal?
4. ¿Qué riesgos de privacidad y sesgo existen al analizar rostros?
5. ¿Cómo modificaría el notebook para guardar únicamente el resultado numérico y eliminar la imagen inmediatamente?